# ARMD：论文与代码对照教程（股票数据）

本 Notebook 对照论文 *Auto-Regressive Moving Diffusion Models for Time Series Forecasting*（AAAI 2025 / arXiv:2412.09328）与仓库实现，使用 **Diffusion-TS `dataset.zip` 中的股票序列**（[`Data/datasets/stock_data.csv`](../Data/datasets/stock_data.csv)）跑通数据处理、训练与推理。

**Protocol note**: this notebook is a lightweight project-import tutorial. It defaults to `Config/stock.yaml`, so it follows the original demo protocol (80/20 window split, L2 loss, `sampling_timesteps=1`, etc.), not the paper-style protocol used by `armd_standalone_full.ipynb` / `Config/stock_paper.yaml` (70/10/20 split, L1 loss, RevIN, `sampling_timesteps=2`). For formula-by-formula study and paper-style comparison, prefer the standalone tutorial; use this notebook to learn the project entry points and import-based workflow.

**环境**：推荐使用 `uv lock && uv sync` 安装 [`pyproject.toml`](../pyproject.toml)；其中 **`torch` 从 PyTorch 官方 CUDA 12.4 索引解析（GPU 版，版本号常带 `+cu124`）**。请在 Jupyter 中选择与本仓库 **`.venv`** 对应的内核。若无 NVIDIA GPU，可按 README「CPU 版 torch」改 `pyproject.toml` 后重新 `uv lock` / `uv sync`。其余说明：不含 `requirements.txt` 里的 GluonTS、MuJoCo 等；完整复现实验请另建环境执行 `pip install -r requirements.txt`。

**运行**：不要用 `uv run notebook`（没有名为 `notebook` 的命令）。请使用例如 `uv run jupyter notebook tutorials/armd_stock_tutorial.ipynb`，或 `uv run python -m notebook …`，或 `uv run jupyter-notebook …`。无头执行见 README 中的 `nbconvert`（建议 `uv run --no-sync ...`，避免每次跑 nbconvert 都重新同步、反复覆盖已安装的 `torch`）。（请在仓库根目录执行；若 cwd 不在本仓库，可设置 **`ARMD_REPO`**。）


## 论文核心思想与代码落点（摘要）

| 论文概念 | 代码位置 | 说明 |
|---------|----------|------|
| 滑动式前向 / 中间状态 | `Models/autoregressive_diffusion/armd.py` → `q_sample` | 对长度为 `window` 的序列按步 `t` 做时间维切片，得到中间状态；**不是**在整段序列上加经典 DDPM 高斯噪声。 |
| 线性型演化（反向） | `Models/autoregressive_diffusion/linear.py` → `Linear` | `nn.Linear` 作用在**时间维**，并用可学习标量 `w[t]` 混合输入与变换结果；训练时另注入 `w_dev[t]*noise`。 |
| 训练目标 | `ARMD._train_loss` | 在滑动后的 `x` 上预测，构造 `target_noise` / `pred_noise` 并加权（`loss_weight`）。 |
| 推理 | `generate_mts` → `fast_sample` / `sample` | 从 batch 的前 `pred_len` 步出发迭代更新；`fast_sample` 中多处 **`sigma=0`、`noise=0`**，接近确定性迭代。 |
| 滑窗数据 | `Utils/Data_utils/real_datasets.py` → `CustomDataset` | `window=192` = **96 历史 + 96 预测**（与 `main.py` 中 `seq_len*2` 一致）；`StandardScaler` 在全部原始数据上 fit。 |

**实现约束（论文未系统展开）**：`armd.py` 顶部 **`pred_len = 96` 为硬编码**，须与 YAML 中 `model.params.seq_length`、数据 `window` 满足 `window = 2 * seq_length` 的关系一致；`Linear` 内也用固定 `96` 步 beta 调度，修改预测长度需多处同步。


---

### 论文未详述：工程实现（实现细节标注）

以下内容在论文中通常**不会逐项写出**，但本仓库依赖它们才能稳定训练与采样：

1. **优化与正则**：`engine/solver.py` — Adam、`clip_grad_norm_=1.0`、**EMA**（`ema-pytorch`）、学习率调度 `ReduceLROnPlateauWithWarmup`（`engine/lr_sch.py`）。
2. **批次共享同一时间步**：`ARMD.forward` 中 `t = torch.randint(0, num_timesteps, (1,)).repeat(b)`，整个 batch 使用**同一个**扩散步 `t`。
3. **DDPM 族缓冲区**：`cosine_beta_schedule`、`posterior_*`、`loss_weight` 等沿袭 `denoising-diffusion-pytorch` 风格；其中一部分与滑动前向组合使用，其余可视为实现模板保留量，阅读时可对照公式区分。
4. **`Trainer.sample` 与 `generate_mts` 签名不一致**：`solver.py` 中无条件采样调用 `generate_mts(batch_size=...)`，而 `ARMD.generate_mts(self, x)` 只接受输入序列；**无条件路径会报错**。本教程仅演示与 `main.py` 一致的 **`sample_forecast`**。
5. **`langevin_fn`**：补缺场景使用，预测主路径不涉及。
6. **本 Notebook 使用最小依赖**：不包含 `requirements.txt` 中与 stock 教程无关的大型包。

---


In [ ]:
# Resolve repo root: nbconvert may use cwd=tutorials/ — walk parents until Models/ exists
from __future__ import annotations

import os
import sys
from pathlib import Path

# If headless run uses a temp cwd, set: ARMD_REPO=/path/to/ARMD (or setx on Windows)
_repo_env = os.environ.get("ARMD_REPO")
_start = Path(_repo_env).resolve() if _repo_env else Path.cwd().resolve()
REPO_ROOT = None
for candidate in [_start, *_start.parents]:
    if (candidate / "Models").is_dir() and (candidate / "Utils").is_dir():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError(
        "Cannot find repo root (directory containing Models/). "
        "Set env ARMD_REPO to your clone path, or os.chdir(path_to_armd)."
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

import random
import numpy as np
import torch

from sklearn.metrics import mean_squared_error, mean_absolute_error

from Utils.io_utils import load_yaml_config, instantiate_from_config
from Data.build_dataloader import build_dataloader, build_dataloader_cond
from engine.solver import Trainer


def set_seed(seed: int = 2023) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(2023)

# torch 安装来源见 pyproject.toml 中 [[tool.uv.index]] / [tool.uv.sources]（CUDA 12.4 预编译包）
print("torch:", torch.__version__)
print("torch.version.cuda (wheel):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "当前为 CPU：请确认已执行 uv lock && uv sync、NVIDIA 驱动已装，且 Notebook 内核为 .venv 中的 Python。"
    )


## 股票 CSV 数据（Diffusion-TS / dataset.zip）

论文与仓库中的 **Stock 基准** 使用 [Diffusion-TS](https://github.com/Y-debug-sys/Diffusion-TS) 发布的数据：从 README 中的 **Google Drive 下载 `dataset.zip`**，解压后将其中的 `stock_data.csv` 放到本仓库 [`Data/datasets/stock_data.csv`](../Data/datasets/stock_data.csv)（与 [README](../README.md) 的 *Dataset Preparation* 一致）。

与 [`Config/stock.yaml`](../Config/stock.yaml) 的对应关系：

- **路径**：`./Data/datasets/stock_data.csv`
- **列**：该 CSV **无日期列**，一般为 6 列：`Open, High, Low, Close, Adj_Close, Volume`。因此配置里 **`name` 不能为 `etth`**（`etth` 会删掉第一列；ETTh 首列是日期，而本文件首列是 **Open**，误删会丢掉变量）。教程已改用 **`name: stock`**，并设 **`feature_size: 6`**。

若本地尚未放置该文件，下一格会报错并提示下载路径；不要用旧的「随机合成」数据复现实验。


In [ ]:
import pandas as pd

DATA_PATH = REPO_ROOT / "Data" / "datasets" / "stock_data.csv"

# Diffusion-TS dataset.zip：6 列数值、无 date；与 stock.yaml（name=stock, feature_size=6）一致
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到 {DATA_PATH.resolve()}。请将 dataset.zip 解压后的 stock_data.csv 放到 Data/datasets/，"
        "参见 README「Dataset Preparation」与 Diffusion-TS 仓库说明。"
    )

df = pd.read_csv(DATA_PATH)
print("数据文件:", DATA_PATH.resolve())
print("形状 (行, 列):", df.shape)
print("列名:", list(df.columns))
print(df.head(3))


## 加载配置并写入教程用超参

为缩短运行时间，覆盖 `max_epochs`、`batch_size`、`save_cycle`，并将 checkpoint 目录改到独立文件夹，避免覆盖你已有的训练结果。

**论文未详述**：优化器、调度器、EMA 等均由 YAML + `Trainer` 固定，非论文公式一部分。


In [ ]:
CONFIG_PATH = REPO_ROOT / "Config" / "stock.yaml"
SAVE_DIR = str(REPO_ROOT / "forecasting_exp_notebook")

configs = load_yaml_config(CONFIG_PATH)

# --- 教程覆盖（保持模型结构不变，仅缩短训练并减小 batch）---
configs["solver"]["max_epochs"] = 40
configs["solver"]["save_cycle"] = 100000  # 演示期间不写 checkpoint
configs["solver"]["results_folder"] = str(REPO_ROOT / "Checkpoints_notebook_stock")
configs["dataloader"]["batch_size"] = 32

print("Config loaded from", CONFIG_PATH)
print("max_epochs =", configs["solver"]["max_epochs"], ", batch_size =", configs["dataloader"]["batch_size"])

# 与 main.py 一致：优先 cuda:0（Trainer 将 batch 放到 model.device）
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)
if getattr(torch.version, "cuda", None) and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA build of torch but cuda.is_available() is False: check NVIDIA driver and Jupyter kernel (.venv)"
    )


## 构建模型

- `ARMD` 中 `seq_length`（如 96）传入 `Linear` 作为 **时间维长度**（`q_sample` 将 `192` 裁成 `96` 后再进网络）。
- `feature_size` 必须与 CSV 数值列维数一致（Diffusion-TS 的 `stock_data.csv` 为 **6**）。
- `main.py` 设置 `model.fast_sampling = True` 以使用步数更少的 `fast_sample`。

**论文未详述**：`sampling_timesteps`（如 stock 中为 1）与 `fast_sample` 中强制 `sigma=0` 属于实现层面的加速与确定性近似。


In [ ]:
class Args:
    # 最小参数集，满足 build_dataloader / Trainer 访问字段

    def __init__(self):
        self.config_path = str(CONFIG_PATH)
        self.save_dir = SAVE_DIR
        self.gpu = 0
        self.mode = "predict"
        self.pred_len = 96
        self.name = "notebook_armd"


import os

os.makedirs(SAVE_DIR, exist_ok=True)

args = Args()

model = instantiate_from_config(configs["model"]).to(DEVICE)
model.fast_sampling = True  # 与 main.py 一致，走 fast_sample

seq_len = 96  # 预测长度 / 模型 seq_length，对应 armd.pred_len
feat_num = configs["model"]["params"]["feature_size"]

print(model)
print("seq_length(model)=", model.seq_length, "feature_size=", feat_num)


## 训练

`Trainer` 内部：`loss = model(data)`（`ARMD.forward` 采样单个 `t` 并调用 `_train_loss`）、反向与 EMA 更新。

**论文未详述**：EMA、梯度裁剪、Adam 超参（`betas=[0.9, 0.96]`）见 `engine/solver.py`。


In [ ]:
dataloader_info = build_dataloader(configs, args)
train_dl = dataloader_info["dataloader"]

trainer = Trainer(config=configs, args=args, model=model, dataloader={"dataloader": train_dl}, logger=None)
trainer.train()
print("训练结束（教程用少量 epoch，非论文完整训练）。")


## 推理与指标

使用 **`Trainer.sample_forecast`**：对每个测试 batch，模型用 `generate_mts(x)`，其中 `x` 为完整窗口 `[B, 192, F]`；内部取 `x[:, :pred_len, :]` 作为迭代起点，输出预测段 `[B, 96, F]`。

与 `main.py` 一致：将预测与 **真实未来段** `x[:, seq_len:, :]`（即后 96 步）比较 MSE/MAE。数值在 **标准化空间**（与 `main.py` 一致）；若要原始量纲需 `scaler.inverse_transform`。

**注意**：不要使用 `Trainer.sample` 做无条件生成（参数与 `generate_mts` 不匹配）。


In [ ]:
args.mode = "predict"
args.pred_len = seq_len

test_info = build_dataloader_cond(configs, args)
test_dataset = test_info["dataset"]
test_dataloader = test_info["dataloader"]

shape = [seq_len, feat_num]

model.eval()
samples, reals = trainer.sample_forecast(test_dataloader, shape=shape)

mse = mean_squared_error(samples.reshape(-1), reals.reshape(-1))
mae = mean_absolute_error(samples.reshape(-1), reals.reshape(-1))
print("MSE (scaled space):", mse)
print("MAE (scaled space):", mae)
print("samples shape:", samples.shape, "reals shape:", reals.shape)


## 小结

- **前向（滑动）**：`q_sample` 按 `t` 切片，把 192 步窗口变为 96 步中间状态，对应论文中沿时间推进的中间变量。
- **网络**：`Linear` 对时间维做线性变换并与 `w[t]` 混合；训练时 `w_dev[t]*noise` 引入随机性。
- **反向采样**：`fast_sample` 在默认配置下接近确定性迭代（噪声项置零）。
- **工程**：EMA / 优化器 / 共享 `t` / checkpoint 路径等为工程细节；无条件 `Trainer.sample` 与当前 `generate_mts` API 不兼容，已在上文标注。

完成后可将 `max_epochs` 调回 `stock.yaml` 原始值做正式实验。


## 线性模型实验

这一节使用**同样的 Stock 数据窗口、同样的归一化、同样的测试目标**做一个线性基线实验。ARMD 使用 `CustomDataset` 产生的窗口形状为 `[N, 192, 6]`，其中前 96 步是历史、后 96 步是预测目标。这里把历史半段 `[96, 6]` 展平成 576 维，用普通最小二乘（OLS）和 Ridge 回归直接预测未来半段的 576 维。

同时加入 Persistence 基线：把最后一个历史观测值重复 96 步。股票序列接近随机游走时，这个简单基线往往很强，可用于判断模型是否真正学到了额外信息。


In [ ]:

from sklearn.linear_model import LinearRegression, Ridge


def split_context_future(samples: np.ndarray, seq_len: int):
    """[N, 2*seq_len, F] -> flattened context, flattened future, future tensor."""
    context = samples[:, :seq_len, :]
    future = samples[:, seq_len:, :]
    n = samples.shape[0]
    return context.reshape(n, -1), future.reshape(n, -1), future


def evaluate_flat(pred, true):
    return (
        float(mean_squared_error(pred.reshape(-1), true.reshape(-1))),
        float(mean_absolute_error(pred.reshape(-1), true.reshape(-1))),
    )


# 复用前面 ARMD 完全相同的 Dataset 实例；若当前会话未保留，则按同一配置重建。
if "dataloader_info" not in globals():
    dataloader_info = build_dataloader(configs, args)
if "test_info" not in globals():
    args.mode = "predict"
    args.pred_len = seq_len
    test_info = build_dataloader_cond(configs, args)

train_samples = np.asarray(dataloader_info["dataset"].samples)
test_samples = np.asarray(test_info["dataset"].samples)
X_train, y_train, _ = split_context_future(train_samples, seq_len)
X_test, y_test, y_test_seq = split_context_future(test_samples, seq_len)

linear_results = []

last_value = test_samples[:, seq_len - 1:seq_len, :]
persistence_pred = np.repeat(last_value, seq_len, axis=1)
linear_results.append(("Persistence (last value)", *evaluate_flat(persistence_pred, y_test_seq)))

ols = LinearRegression()
ols.fit(X_train, y_train)
ols_pred = ols.predict(X_test)
linear_results.append(("Linear regression (OLS)", *evaluate_flat(ols_pred, y_test)))

ridge_alpha = 1.0
ridge = Ridge(alpha=ridge_alpha)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
linear_results.append((f"Linear regression (Ridge alpha={ridge_alpha:g})", *evaluate_flat(ridge_pred, y_test)))

if "mse" in globals() and "mae" in globals():
    linear_results.insert(0, ("ARMD (notebook run)", float(mse), float(mae)))

print(f"train windows: {train_samples.shape}  test windows: {test_samples.shape}")
print(f"{'model':<42}{'MSE':>12}{'MAE':>12}")
print("-" * 66)
for name, mse_i, mae_i in linear_results:
    print(f"{name:<42}{mse_i:>12.4f}{mae_i:>12.4f}")


## 线性模型预测可视化

下面随机抽取几个测试窗口，对比真实未来、OLS 预测，以及前面已经算出的 ARMD 预测。横轴虚线左侧是历史输入，右侧是需要预测的未来 96 步。


In [ ]:

import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n_show = min(4, test_samples.shape[0])
idx_list = rng.choice(test_samples.shape[0], size=n_show, replace=False)
feat_show = min(2, test_samples.shape[-1])

ols_seq = ols_pred.reshape(test_samples.shape[0], seq_len, test_samples.shape[-1])
armd_available = "samples" in globals() and samples is not None and samples.shape[0] == test_samples.shape[0]

fig, axes = plt.subplots(n_show, feat_show, figsize=(6 * feat_show, 2.8 * n_show), sharex=True)
if n_show == 1:
    axes = np.array([axes])
if feat_show == 1:
    axes = axes[:, None]

for r, idx in enumerate(idx_list):
    for c in range(feat_show):
        ax = axes[r, c]
        xh = np.arange(seq_len)
        xf = np.arange(seq_len, 2 * seq_len)
        ax.plot(xh, test_samples[idx, :seq_len, c], color="#555", lw=0.9,
                label="history" if (r == 0 and c == 0) else None)
        ax.plot(xf, y_test_seq[idx, :, c], color="#1f77b4", lw=1.0,
                label="true future" if (r == 0 and c == 0) else None)
        ax.plot(xf, ols_seq[idx, :, c], color="#2ca02c", lw=1.1, ls="--",
                label="OLS" if (r == 0 and c == 0) else None)
        if armd_available:
            ax.plot(xf, samples[idx, :, c], color="#d62728", lw=1.1, ls=":",
                    label="ARMD" if (r == 0 and c == 0) else None)
        ax.axvline(seq_len - 0.5, color="grey", ls="--", lw=0.8)
        ax.grid(alpha=0.3)
        ax.set_title(f"test window #{int(idx)} feat{c}", fontsize=9)
        if c == 0:
            ax.set_ylabel("z-score", fontsize=8)

axes[0, 0].legend(loc="upper left", fontsize=8)
fig.suptitle("Linear baseline vs true future", y=1.01)
plt.tight_layout()
plt.show()
